# Part 3: LSTM

**Owner:** Parfait Christian Henry UHIRIWE

This notebook is my part of the group project on vaccine sentiment, and it builds the second recurrent model in the comparison: an embedding layer, an LSTM layer and a single linear output that predicts where a tweet sits on the sentiment scale from -1 (anti-vaccine) to +1 (pro-vaccine). The question for this part is whether the memory cell of an LSTM, which can keep or forget information as it reads a tweet, helps more than the plain recurrent state of the Simple RNN. To keep that question fair, the LSTM uses the shared split, the text cleaning Fred chose in notebook 01 and the same vectorizer settings as the Simple RNN, so the main thing that changes between the two notebooks is the recurrent cell itself.

| Output | Path |
|---|---|
| Validation predictions | `results/lstm_validation_predictions.csv` |
| Experiment log | `results/experiments/lstm_experiments.csv` |
| Figures | `results/figures/lstm_*.png` |
| Final model | `models/lstm.keras` (+ `models/lstm_vectorizer.json`) |

## 1. Setup

In [ ]:
# Setup: works on Google Colab and locally.
import subprocess
import sys
from pathlib import Path

REPO = "vaccine-sentiment-sequential-models"
if "google.colab" in sys.modules:
    ROOT = Path("/content") / REPO
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/Adossi-design/{REPO}.git", str(ROOT)], check=True)
    else:  # get teammates' latest pushes (e.g. the split files)
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))

from src import common

common.print_environment()
common.set_seed()

The data is not stored in the repository, so the next cell makes sure `data/Train.csv` is available. On Colab it asks for the file to be uploaded, and in both cases it checks the file against the official version so that every model is trained on identical data.

In [ ]:
common.fetch_train_csv()

In [ ]:
import json

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import layers

from src.preprocess import prepare_series

RESULTS_DIR = Path(common.RESULTS_DIR)
FIGURES_DIR = Path(common.FIGURES_DIR)
MODELS_DIR = Path(common.MODELS_DIR)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OWNER = "Parfait Christian Henry UHIRIWE"

## 2. Preparing the Data

The training and validation rows come from the shared split through `common.load_train_validation()`, which also checks that the two ID files do not overlap and that every row has a valid label. The text then goes through `prepare_series()`, which is the cleaning Fred decided on in notebook 01, so the LSTM sees tweets prepared in exactly the same way as every other model.

In [ ]:
train_df, val_df = common.load_train_validation()
train_text = np.asarray(prepare_series(train_df["safe_text"]), dtype=str)
val_text = np.asarray(prepare_series(val_df["safe_text"]), dtype=str)
y_train = train_df["label"].to_numpy(dtype="float32")
y_val = val_df["label"].to_numpy(dtype="float32")

print(f"Training tweets: {len(train_df)}, validation tweets: {len(val_df)}")
print(train_df["label"].value_counts().sort_index())

The Simple RNN notebook already chose the vocabulary size and sequence length from the training split and saved them in `models/simple_rnn_vectorizer.json`. I reuse those settings rather than choosing new ones, because giving both models the same token input is what makes it possible to attribute a difference in their results to the recurrent cell instead of to the input. The cell below also recomputes the length statistics on the cleaned training text, so the reused sequence length can be checked against the data rather than taken on trust.

In [ ]:
with open(MODELS_DIR / "simple_rnn_vectorizer.json") as f:
    rnn_vectorizer_settings = json.load(f)

MAX_TOKENS = rnn_vectorizer_settings["max_tokens"]
MAX_LEN = rnn_vectorizer_settings["max_len"]

token_counts = np.array([len(text.split()) for text in train_text])
print(f"Reused settings: max_tokens={MAX_TOKENS}, max_len={MAX_LEN}")
print(f"Median length: {np.median(token_counts):.0f} tokens, 99th percentile: {np.percentile(token_counts, 99):.0f} tokens")
print(f"Training tweets longer than max_len: {(token_counts > MAX_LEN).mean():.2%}")

The vectorizer uses `standardize=None` and splits on whitespace because `prepare_series()` has already cleaned the text, and the default Keras standardiser would remove the hashtags, punctuation and emojis that notebook 01 decided to keep. It is adapted on the training tweets only, so nothing about the validation vocabulary reaches the model.

In [ ]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize=None,
    split="whitespace",
    output_sequence_length=MAX_LEN,
)
vectorizer.adapt(train_text)

X_train = keras.ops.convert_to_numpy(vectorizer(train_text))
X_val = keras.ops.convert_to_numpy(vectorizer(val_text))
print(f"Vocabulary size: {vectorizer.vocabulary_size()}, input shapes: {X_train.shape}, {X_val.shape}")

## 3. Building the Model

The network reads the token IDs through an embedding layer, passes them through one LSTM layer and ends with a single linear unit, because the target is a continuous score scored with RMSE. The embedding uses `mask_zero=True` so that the padding added to short tweets does not flow through the memory cell as if it were text. Dropout is applied through the LSTM's own `dropout` argument, which keeps the fast cuDNN kernel on a GPU, whereas `recurrent_dropout` would switch to a much slower implementation. The bidirectional option is kept in the same function because one experiment tests it, and it still uses the same LSTM cell.

In [ ]:
def build_lstm(embedding_dim, units, dropout, learning_rate, bidirectional=False):
    """Build an embedding, LSTM and linear output network that predicts a tweet sentiment score."""
    inputs = keras.Input(shape=(MAX_LEN,), dtype="int64")
    x = layers.Embedding(vectorizer.vocabulary_size(), embedding_dim, mask_zero=True)(inputs)
    lstm_layer = layers.LSTM(units, dropout=dropout)
    x = layers.Bidirectional(lstm_layer)(x) if bidirectional else lstm_layer(x)
    outputs = layers.Dense(1)(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


build_lstm(embedding_dim=64, units=64, dropout=0.0, learning_rate=1e-3).summary()

## 4. Training and Logging Runs

Every run goes through `train_lstm()`, so the seed, early stopping and the RMSE calculation stay identical between experiments. Early stopping watches the validation loss and restores the best weights, which stops training once the model starts fitting noise in the training set. The validation RMSE is computed with `common.rmse`, as the group conventions require, and the predictions are not clipped to [-1, 1] because the group left that decision to notebook 05 so it can be applied to all five models together.

Logging is a separate step on purpose. `log_run()` refuses to write a run until its observation has been written, so every row in the experiment log contains a note that was made after looking at the actual result.

In [ ]:
BASE_CONFIG = {
    "embedding_dim": 64,
    "units": 64,
    "dropout": 0.0,
    "learning_rate": 1e-3,
    "batch_size": 32,
    "epochs": 30,
    "patience": 3,
    "bidirectional": False,
    "agreement_weights": False,
}
runs = {}


def train_lstm(run_id, config, seed=42):
    """Train one LSTM configuration on the training split and return its validation results."""
    common.set_seed(seed)
    model = build_lstm(config["embedding_dim"], config["units"], config["dropout"], config["learning_rate"], config["bidirectional"])
    sample_weight = train_df["agreement"].to_numpy(dtype="float32") if config["agreement_weights"] else None
    early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=config["patience"], restore_best_weights=True)

    history = model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
        validation_data=(X_val, y_val),
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        callbacks=[early_stopping],
        verbose=0,
    )
    y_pred = model.predict(X_val, verbose=0).ravel()

    result = {
        "run_id": run_id,
        "seed": seed,
        "config": dict(config),
        "model": model,
        "history": history.history,
        "y_pred": y_pred,
        "val_rmse": common.rmse(val_df["label"], y_pred),
        "best_epoch": int(np.argmin(history.history["val_loss"])) + 1,
    }
    runs[run_id] = result
    print(f"{run_id} (seed {seed}): validation RMSE {result['val_rmse']:.4f}, best epoch {result['best_epoch']} of {len(history.history['loss'])}")
    return result


def log_run(result, change_from_previous, observation):
    """Record one finished run in the LSTM experiment log once its observation has been written."""
    if not observation:
        raise ValueError(f"Write the observation for {result['run_id']} after reading its result, then run this cell again.")
    config = result["config"]
    common.log_experiment(
        run_id=result["run_id"],
        owner=OWNER,
        model="lstm",
        approach="Embedding -> Bidirectional LSTM -> Dense(1)" if config["bidirectional"] else "Embedding -> LSTM -> Dense(1)",
        input_representation=f"token IDs from TextVectorization (max_tokens={MAX_TOKENS}, max_len={MAX_LEN}, same settings as the Simple RNN)",
        preprocessing="agreed notebook 01 text preparation (prepare_series)",
        hyperparameters={**config, "seed": result["seed"], "best_epoch": result["best_epoch"]},
        change_from_previous=change_from_previous,
        val_rmse=result["val_rmse"],
        observation=observation,
    )


def plot_learning_curve(result, title, save_name=None):
    """Plot the training and validation RMSE of one run across epochs."""
    plt.figure(figsize=(7, 4))
    plt.plot(result["history"]["rmse"], label="training RMSE")
    plt.plot(result["history"]["val_rmse"], label="validation RMSE")
    plt.axvline(result["best_epoch"] - 1, color="grey", linestyle="--", label="restored epoch")
    plt.xlabel("epoch")
    plt.ylabel("RMSE")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    if save_name:
        plt.savefig(FIGURES_DIR / save_name, dpi=150)
    plt.show()

## 5. Experiments

### 5.1 Starting configuration

The first run uses moderate settings with no dropout, so that it gives a clear reference point and shows whether the LSTM overfits before any regularisation is added.

In [ ]:
lstm_01 = train_lstm("lstm-01", BASE_CONFIG)
plot_learning_curve(lstm_01, "lstm-01: starting configuration")

In [ ]:
log_run(
    lstm_01,
    change_from_previous="first run: reference LSTM on the same token input as the Simple RNN",
    observation=None,
)

### 5.2 How much do results move between seeds?

A recurrent network can give a different RMSE from a different random start even when nothing else changes. Before judging any change, the starting configuration is trained with two more seeds, and the spread of the three results gives the size of a difference that could appear by chance. A later change only counts as an improvement if it beats the starting configuration by more than this spread.

In [ ]:
noise_runs = [lstm_01] + [train_lstm(f"lstm-01-seed{seed}", BASE_CONFIG, seed=seed) for seed in (43, 44)]
noise_rmse = np.array([run["val_rmse"] for run in noise_runs])
baseline_mean = float(noise_rmse.mean())
seed_noise = float(noise_rmse.std(ddof=1))
print(f"Starting configuration over three seeds: mean RMSE {baseline_mean:.4f}, standard deviation {seed_noise:.4f}")

In [ ]:
noise_observations = {
    "lstm-01-seed43": None,
    "lstm-01-seed44": None,
}
for run in noise_runs[1:]:
    log_run(run, "seed check: same configuration as lstm-01 with a different seed", noise_observations[run["run_id"]])

### 5.3 One change at a time

Each run below changes a single setting from the starting configuration, so that any difference in RMSE can be linked to that one setting. The reason for each change is written next to it.

The last run weights each training tweet by its annotator agreement. The agreement column is still not a model input, since the model never sees it and it is not needed at prediction time. It only reduces how much the low-agreement tweets pull on the training loss, which tests whether noisy labels are limiting the LSTM.

In [ ]:
one_change_runs = {
    "lstm-02": ({"dropout": 0.3}, "adds input dropout of 0.3 to test whether the starting LSTM overfits the small training set"),
    "lstm-03": ({"units": 128}, "doubles the LSTM units to 128 to test whether a larger memory cell helps"),
    "lstm-04": ({"embedding_dim": 32}, "halves the embedding to 32, the size the Simple RNN finally used, to test whether a smaller embedding is enough"),
    "lstm-05": ({"bidirectional": True}, "reads each tweet in both directions to test whether context from later words, such as a late negation, helps"),
    "lstm-06": ({"agreement_weights": True}, "weights training tweets by annotator agreement to test whether uncertain labels limit the model"),
}

change_results = {run_id: train_lstm(run_id, {**BASE_CONFIG, **change}) for run_id, (change, _) in one_change_runs.items()}

comparison = pd.DataFrame(
    [{"run_id": run_id, "val_rmse": result["val_rmse"], "best_epoch": result["best_epoch"]} for run_id, result in change_results.items()]
)
comparison["improvement_over_mean"] = baseline_mean - comparison["val_rmse"]
comparison["beats_seed_noise"] = comparison["improvement_over_mean"] > seed_noise
print(f"Starting mean {baseline_mean:.4f}, seed noise {seed_noise:.4f}")
print(comparison[["run_id", "val_rmse", "improvement_over_mean", "beats_seed_noise", "best_epoch"]].round(4).to_string(index=False))

In [ ]:
change_observations = {
    "lstm-02": None,
    "lstm-03": None,
    "lstm-04": None,
    "lstm-05": None,
    "lstm-06": None,
}
for run_id, result in change_results.items():
    log_run(result, one_change_runs[run_id][1], change_observations[run_id])

In [ ]:
plt.figure(figsize=(8, 4))
labels = ["lstm-01 (mean)"] + list(comparison["run_id"])
values = [baseline_mean] + list(comparison["val_rmse"])
plt.bar(labels, values, color=["grey"] + ["tab:blue"] * len(comparison))
plt.axhspan(baseline_mean - seed_noise, baseline_mean + seed_noise, color="grey", alpha=0.2, label="starting mean +/- seed noise")
plt.ylim(min(values) - 0.02, max(values) + 0.02)
plt.ylabel("validation RMSE")
plt.title("LSTM experiments")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_experiments.png", dpi=150)
plt.show()

### 5.4 Final configuration

The final configuration keeps only the changes that beat the seed noise in the table above. If no change clearly helps, the starting configuration is kept, since adding settings that make no measurable difference would only make the model harder to explain. The chosen configuration is then trained on three fresh seeds that were not used for any decision, which checks that the result was not a lucky run.

In [ ]:
# Add only the changes that beat the seed noise, for example {"dropout": 0.3}.
final_changes = {}
FINAL_CONFIG = {**BASE_CONFIG, **final_changes}

final_run = train_lstm("lstm-final", FINAL_CONFIG, seed=42)
fresh_runs = [train_lstm(f"lstm-final-seed{seed}", FINAL_CONFIG, seed=seed) for seed in (7, 123, 2026)]
fresh_rmse = np.array([run["val_rmse"] for run in fresh_runs])
print(f"Final configuration on fresh seeds: mean RMSE {fresh_rmse.mean():.4f}, standard deviation {fresh_rmse.std(ddof=1):.4f}")

In [ ]:
final_observations = {
    "lstm-final": None,
    "lstm-final-seed7": None,
    "lstm-final-seed123": None,
    "lstm-final-seed2026": None,
}
final_change = f"final configuration: starting configuration plus {final_changes}" if final_changes else "final configuration: starting configuration, since no change beat the seed noise"
for run in [final_run] + fresh_runs:
    log_run(run, final_change, final_observations[run["run_id"]])

The seed 42 run of the final configuration is the official LSTM result, which matches the seed used across the group. Its learning curve, predictions, model and vectorizer settings are saved below.

In [ ]:
plot_learning_curve(final_run, "Final LSTM", save_name="lstm_learning_curve.png")

y_pred = final_run["y_pred"]
common.save_predictions("lstm", val_df, y_pred)
final_run["model"].save(MODELS_DIR / "lstm.keras")

with open(MODELS_DIR / "lstm_vectorizer.json", "w") as f:
    json.dump(
        {
            "standardize": "none (prepare_series already cleans the text)",
            "split": "whitespace",
            "max_tokens": MAX_TOKENS,
            "max_len": MAX_LEN,
            "adapted_on": "shared training split (splits/train_ids.csv)",
            "preprocessing": "prepare_series() from src/preprocess.py: the shared cleaning chosen in notebook 01 from the EDA",
        },
        f,
        indent=2,
    )

## 6. Error Analysis

RMSE compresses every error into one number, so this section looks at where the errors come from. The first cell splits the RMSE by true label, because an overall score can hide a model that does well on the common classes and badly on a rare one.

In [ ]:
analysis = val_df[["tweet_id", "safe_text", "label", "agreement"]].copy()
analysis["lstm_pred"] = y_pred
analysis["abs_error"] = (analysis["lstm_pred"] - analysis["label"]).abs()

label_rows = [
    {"label": label, "tweets": len(group), "rmse": common.rmse(group["label"], group["lstm_pred"]), "mean_prediction": group["lstm_pred"].mean()}
    for label, group in analysis.groupby("label")
]
print(pd.DataFrame(label_rows).round(4).to_string(index=False))

plt.figure(figsize=(6, 4))
plt.boxplot([analysis.loc[analysis["label"] == label, "lstm_pred"] for label in (-1, 0, 1)])
plt.xticks([1, 2, 3], ["-1", "0", "1"])
plt.xlabel("true label")
plt.ylabel("LSTM prediction")
plt.title("LSTM predictions by true label")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_predictions_by_label.png", dpi=150)
plt.show()

Rounding the predictions to the nearest label turns them into classes, and the cross-table below shows which mistakes the model makes. Confusing a neutral tweet with a neighbouring class is a much smaller problem than calling an anti-vaccine tweet pro-vaccine. The rounding is only used to read the errors and does not replace RMSE.

In [ ]:
analysis["rounded_pred"] = analysis["lstm_pred"].round().clip(-1, 1).astype(int)
print(pd.crosstab(analysis["label"], analysis["rounded_pred"], rownames=["true"], colnames=["rounded prediction"]))

The next cell checks whether the largest errors are concentrated on tweets where the annotators disagreed. If they are, part of the remaining error may come from uncertain labels rather than from the model.

In [ ]:
largest = analysis.nlargest(15, "abs_error")
print(f"Mean agreement, all validation tweets: {analysis['agreement'].mean():.3f}")
print(f"Mean agreement, 15 largest errors: {largest['agreement'].mean():.3f}")
print(f"Full-agreement tweets among the 15 largest errors: {(largest['agreement'] == 1).sum()}")
print(analysis.groupby("agreement")["abs_error"].agg(["count", "mean"]).round(4))

These are the 15 tweets with the largest errors. Each should be read before its error is explained, and where the text gives no clear reason, it is better to say so than to force it into a category.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    print(largest[["tweet_id", "label", "lstm_pred", "agreement", "safe_text"]].round(3).to_string(index=False))

## 7. Comparison with the Simple RNN

Both models use the same split, cleaning and token input, so this comparison isolates the effect of the recurrent cell. The Simple RNN's largest errors were negative tweets predicted as positive, several of which contained negation. A memory cell should in principle carry a negation forward to the words it affects, so the cell below measures the error on tweets with and without negation words for both models, rather than assuming the LSTM handles them better.

In [ ]:
rnn_predictions = pd.read_csv(RESULTS_DIR / "rnn_validation_predictions.csv")[["tweet_id", "predicted_label"]]
both = analysis.merge(rnn_predictions.rename(columns={"predicted_label": "rnn_pred"}), on="tweet_id", how="inner")
assert len(both) == len(analysis), "The Simple RNN predictions do not cover the same validation tweets."

print(f"Validation RMSE, Simple RNN: {common.rmse(both['label'], both['rnn_pred']):.4f}")
print(f"Validation RMSE, LSTM: {common.rmse(both['label'], both['lstm_pred']):.4f}")

both["has_negation"] = both["safe_text"].str.lower().str.contains(r"\b(?:not|no|never|nor|cannot)\b|n't", regex=True)
rows = []
for (label, has_negation), group in both.groupby(["label", "has_negation"]):
    rows.append({"label": label, "has_negation": has_negation, "tweets": len(group), "rnn_rmse": common.rmse(group["label"], group["rnn_pred"]), "lstm_rmse": common.rmse(group["label"], group["lstm_pred"])})
print(pd.DataFrame(rows).round(4).to_string(index=False))

both["rnn_abs_error"] = (both["rnn_pred"] - both["label"]).abs()
rnn_top = set(both.nlargest(50, "rnn_abs_error")["tweet_id"])
lstm_top = set(both.nlargest(50, "abs_error")["tweet_id"])
print(f"Tweets in both models' 50 largest errors: {len(rnn_top & lstm_top)}")

## 8. Notes and Sources

Literature used for the decisions in this notebook:

- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735-1780. The LSTM architecture and its memory cell.
- Bengio, Y., Simard, P., & Frasconi, P. (1994). Learning long-term dependencies with gradient descent is difficult. *IEEE Transactions on Neural Networks, 5*(2), 157-166. Why a plain RNN struggles to carry information across a sequence, which motivates comparing it with the LSTM.
- Gers, F. A., Schmidhuber, J., & Cummins, F. (2000). Learning to forget: Continual prediction with LSTM. *Neural Computation, 12*(10), 2451-2471. The forget gate used in modern LSTM layers.
- Schuster, M., & Paliwal, K. K. (1997). Bidirectional recurrent neural networks. *IEEE Transactions on Signal Processing, 45*(11), 2673-2681. The basis for experiment lstm-05.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research, 15*, 1929-1958. The basis for experiment lstm-02.
- Müller, M. M., & Salathé, M. (2019). Crowdbreaks: Tracking health trends using public social media data and crowdsourcing. *Frontiers in Public Health, 7*, 81. The source of the tweets and the three-annotator labels.